# Meta DS Interview Practice — Facebook Marketplace

## Context
You're a Data Scientist on the Facebook Marketplace team. You've been given a table of daily listing performance data. Your job is to analyze seller and category health — just like you would in a real product review.

Work through each question below. Think out loud in comments as you go — interviewers want to follow your reasoning, not just see the final answer.

---

## The Dataset

| Column | Description |
|---|---|
| `listing_id` | Unique ID for each Marketplace listing |
| `seller_id` | ID of the seller who posted the listing |
| `category` | Product category (Electronics, Furniture, Clothing, Vehicles, Home & Garden) |
| `date` | Date of the record (daily grain) |
| `views` | Number of times the listing was viewed |
| `messages` | Number of buyer messages sent about the listing |
| `saves` | Number of users who saved the listing |
| `transactions` | Number of completed sales |
| `listing_price` | Price of the item in USD |

---

## Questions

### Q1 — Warm-up: Listing Engagement Metrics
Calculate the **message rate** and **save rate** for each category across all dates. Return categories ranked by message rate.

- Message Rate = messages / views
- Save Rate = saves / views

---

### Q2 — Aggregation & Grouping: Seller Performance
For each seller, calculate their **total revenue** (transactions × avg listing_price), **total transactions**, and **view-to-transaction rate**.
Flag any seller whose view-to-transaction rate is **more than 2 standard deviations below the mean** — these are struggling sellers.

- View-to-Transaction Rate = transactions / views

---

### Q3 — Time Series: Trend Analysis
The product team wants to understand weekly trends. Resample the data to a **weekly grain** and compute total views, messages, saves, and transactions per week.
Then calculate **week-over-week (WoW) % change** in transactions.

---

### Q4 — Diagnosis: Drop Investigation
A PM comes to you and says: 'Our overall transaction rate looks like it dipped the last two weeks — can you figure out what's going on?'

Using the data, **identify which specific categories or sellers are driving the drop**. Walk through your approach in comments before writing the code.

---

### Bonus — Product Question (No Code)
After your analysis, the PM asks:

> 'We are seeing high message rates and save rates, but low transaction rates across several listings. What hypotheses would you form, and how would you investigate further?'

Write your answer as markdown in the cell below. Think about: pricing, trust & safety signals, buyer friction, seller responsiveness, and what additional data you would want.

---

In [1]:
import pandas as pd
import numpy as np

np.random.seed(99)

# --- Config ---
n_listings = 25
n_sellers = 8
categories = ['Electronics', 'Furniture', 'Clothing', 'Vehicles', 'Home & Garden']
date_range = pd.date_range(start='2024-10-01', end='2024-11-30', freq='D')

records = []

for listing_id in range(1, n_listings + 1):
    seller_id = (listing_id % n_sellers) + 1
    category = categories[listing_id % len(categories)]
    listing_price = round(np.random.uniform(20, 1500), 2)
    base_views = np.random.randint(200, 2000)
    base_msg_rate = np.random.uniform(0.03, 0.15)
    base_save_rate = np.random.uniform(0.05, 0.20)
    base_txn_rate = np.random.uniform(0.005, 0.04)

    for date in date_range:
        # Inject a transaction rate drop for 2 listings in the last two weeks
        txn_multiplier = 1.0
        if listing_id in [4, 9] and date >= pd.Timestamp('2024-11-17'):
            txn_multiplier = 0.15  # sharp transaction drop

        views = int(base_views * np.random.uniform(0.85, 1.15))
        messages = int(views * base_msg_rate * np.random.uniform(0.9, 1.1))
        saves = int(views * base_save_rate * np.random.uniform(0.9, 1.1))
        transactions = int(views * base_txn_rate * txn_multiplier * np.random.uniform(0.8, 1.2))

        records.append({
            'listing_id': f'L{listing_id:03d}',
            'seller_id': f'S{seller_id:02d}',
            'category': category,
            'date': date,
            'views': views,
            'messages': messages,
            'saves': saves,
            'transactions': transactions,
            'listing_price': listing_price
        })

df = pd.DataFrame(records)
df['date'] = pd.to_datetime(df['date'])

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Listings: {df["listing_id"].nunique()}, Sellers: {df["seller_id"].nunique()}, Categories: {df["category"].nunique()}')
df.head(10)

Dataset shape: (1525, 9)
Date range: 2024-10-01 to 2024-11-30
Listings: 25, Sellers: 8, Categories: 5


,listing_id,seller_id,category,date,views,messages,saves,transactions,listing_price
0,L001,S02,Furniture,2024-10-01,1279,181,207,37,1014.97
1,L001,S02,Furniture,2024-10-02,1557,197,238,48,1014.97
2,L001,S02,Furniture,2024-10-03,1406,176,230,39,1014.97
3,L001,S02,Furniture,2024-10-04,1337,178,222,36,1014.97
4,L001,S02,Furniture,2024-10-05,1370,186,228,36,1014.97
5,L001,S02,Furniture,2024-10-06,1588,202,258,38,1014.97
6,L001,S02,Furniture,2024-10-07,1459,181,213,43,1014.97
7,L001,S02,Furniture,2024-10-08,1234,170,191,31,1014.97
8,L001,S02,Furniture,2024-10-09,1226,146,204,34,1014.97
9,L001,S02,Furniture,2024-10-10,1329,162,231,37,1014.97


---
## Your Work Starts Here
### Q1 — Warm-up: Listing Engagement Metrics
Calculate the **message rate** and **save rate** for each category across all dates. Return categories ranked by message rate.

- Message Rate = messages / views
- Save Rate = saves / views

In [3]:
# your code here

df["message_rate"] = df["messages"] / df["views"]
df["save_rate"] = df["saves"] / df["views"]
#df.head()

# get the average
df.groupby("category").agg(average_message_rate=("message_rate", "mean"),
                           average_save_rate=("save_rate", "mean")).head()

,average_message_rate,average_save_rate
category,,
Clothing,0.075456,0.128513
Electronics,0.087245,0.150806
Furniture,0.099605,0.152646
Home & Garden,0.078421,0.137291
Vehicles,0.110952,0.117542


### Q2 — Aggregation & Grouping: Seller Performance
For each seller, calculate their **total revenue** (transactions × avg listing_price), **total transactions**, and **view-to-transaction rate**.
Flag any seller whose view-to-transaction rate is **more than 2 standard deviations below the mean** — these are struggling sellers.

- View-to-Transaction Rate = transactions / views

In [9]:
seller_stats = df.groupby("seller_id").agg(
    total_views=("views", "sum"),
    total_transactions=("transactions", "sum")
).reset_index()
seller_stats["rate"] = seller_stats["total_transactions"] / seller_stats["total_views"]
seller_stats.head()

,seller_id,total_views,total_transactions,rate
0,S01,201174,4970,0.024705
1,S02,271461,5531,0.020375
2,S03,182389,3765,0.020643
3,S04,236744,5787,0.024444
4,S05,284927,4756,0.016692


In [12]:
threshold = np.mean(seller_stats["rate"]) - 2 *np.std(seller_stats["rate"])
seller_stats["maks"] = seller_stats["rate"] < threshold
seller_stats.head()

,seller_id,total_views,total_transactions,rate,maks
0,S01,201174,4970,0.024705,False
1,S02,271461,5531,0.020375,False
2,S03,182389,3765,0.020643,False
3,S04,236744,5787,0.024444,False
4,S05,284927,4756,0.016692,False


In [ ]:
df["view-to-transaction-rate"] = df["transaction"] / df["views"]


### Q3 — Time Series: Trend Analysis
The product team wants to understand weekly trends. Resample the data to a **weekly grain** and compute total views, messages, saves, and transactions per week.
Then calculate **week-over-week (WoW) % change** in transactions.

In [15]:
df_weekly.head()

,listing_id,seller_id,category,views,messages,saves,transactions,listing_price,message_rate,save_rate,revenue
date,,,,,,,,,,,
2024-10-06,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,182123,17225,25255,3780,110299.62,13.548376,20.641796,3137324.18
2024-10-13,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,208746,19597,28924,4341,128682.89,15.764025,24.055402,3603332.02
2024-10-20,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,209830,19830,28966,4411,128682.89,15.874061,24.079238,3669860.89
2024-10-27,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,210096,19862,28906,4457,128682.89,15.808225,23.954182,3722439.92
2024-11-03,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,211640,20067,29172,4335,128682.89,15.849245,23.983489,3598240.05


In [16]:
# your code here
df_weekly = df.resample('W', on='date').sum()
df_weekly['transactions_change'] = df_weekly['transactions'].pct_change()
df_weekly.head()

,listing_id,seller_id,category,views,messages,saves,transactions,listing_price,message_rate,save_rate,revenue,transactions_change
date,,,,,,,,,,,,
2024-10-06,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,182123,17225,25255,3780,110299.62,13.548376,20.641796,3137324.18,NaN
2024-10-13,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,208746,19597,28924,4341,128682.89,15.764025,24.055402,3603332.02,0.148413
2024-10-20,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,209830,19830,28966,4411,128682.89,15.874061,24.079238,3669860.89,0.016125
2024-10-27,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,210096,19862,28906,4457,128682.89,15.808225,23.954182,3722439.92,0.010428
2024-11-03,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,211640,20067,29172,4335,128682.89,15.849245,23.983489,3598240.05,-0.027373


### Q4 — Diagnosis: Drop Investigation
A PM comes to you and says: 'Our overall transaction rate looks like it dipped the last two weeks — can you figure out what's going on?'

Using the data, **identify which specific categories or sellers are driving the drop**. Walk through your approach in comments before writing the code.

In [18]:
# your code here
df_weekly['views_change'] = df_weekly['views'].pct_change()
df_weekly.tail()

,listing_id,seller_id,category,views,messages,saves,transactions,listing_price,message_rate,save_rate,revenue,transactions_change,views_change
date,,,,,,,,,,,,,
2024-11-03,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,211640,20067,29172,4335,128682.89,15.849245,23.983489,3598240.05,-0.027373,0.007349
2024-11-10,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,209818,19731,29108,4366,128682.89,15.714997,24.144441,3628273.74,0.007151,-0.008609
2024-11-17,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,212950,20156,29479,4437,128682.89,15.899395,24.108195,3683365.54,0.016262,0.014927
2024-11-24,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,209317,19863,28638,4160,128682.89,15.823802,23.875076,3422628.99,-0.062430,-0.017060
2024-12-01,L001L002L003L004L005L006L007L008L009L010L011L0...,S02S03S04S05S06S07S08S01S02S03S04S05S06S07S08S...,FurnitureClothingVehiclesHome & GardenElectron...,182306,17117,25189,3644,110299.62,13.479702,20.631184,2995480.22,-0.124038,-0.129044


### Bonus — Product Question Response

The PM asks: 'We are seeing high message rates and save rates, but low transaction rates across several listings. What hypotheses would you form, and how would you investigate further?'

> Your answer here...